In [1]:
import pandas as pd
import os
from os import listdir, stat
import numpy as np

In [2]:
MIN_SIZE = 512

In [3]:
plantlist = ['Gro_kraftwerk_Mannheim', 'Knapsack_Gas_I']

In [4]:
YEARS = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

In [5]:
YEAR = 2023

In [6]:
files1 = [f for f in os.listdir('.') if os.path.isfile(f)]
files2 = [f for f in files1 if stat(f).st_size > MIN_SIZE]
FILES = list(filter(lambda x: True if x[-4:] == '.csv' else False, files2))

In [7]:
FILES

['Knapsack_Gas_I_202401010000_202412312359_Stunde.csv',
 'Knapsack_Gas_II_202401010000_202412312359_Stunde.csv',
 'Gro_kraftwerk_Mannheim_202101010000_202112312359_Stunde.csv',
 'Knapsack_Gas_I_202301010000_202312312359_Stunde.csv',
 'Gro_kraftwerk_Mannheim_201801010000_201812312359_Stunde(2).csv',
 'Knapsack_Gas_II_201601010000_201612312359_Stunde.csv',
 'Knapsack_Gas_I_202001010000_202012312359_Stunde.csv',
 'Knapsack_Gas_II_202201010000_202212312359_Stunde.csv',
 'Gro_kraftwerk_Mannheim_201501010000_201512312359_Stunde.csv',
 'Gro_kraftwerk_Mannheim_201601010000_201612312359_Stunde(2).csv',
 'Gro_kraftwerk_Mannheim_202201010000_202212312359_Stunde(2).csv',
 'Gro_kraftwerk_Mannheim_201701010000_201712312359_Stunde(2).csv',
 'Gro_kraftwerk_Mannheim_202301010000_202312312359_Stunde(2).csv',
 'Gro_kraftwerk_Mannheim_202401010000_202412312359_Stunde(2).csv',
 'Knapsack_Gas_II_202301010000_202312312359_Stunde.csv',
 'Gro_kraftwerk_Mannheim_202401010000_202412312359_Stunde.csv',
 'Gro_kraf

In [8]:
def read_df(plantname, year):
    return pd.read_csv(plantname + "_" + str(year) + "01010000_" + str(year+1) + "12312359_Stunde.csv", delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')

def read_df(f):
    return pd.read_csv(f, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip')

In [9]:
dfdict = {}
for plant in plantlist:
    df = pd.DataFrame()
    dflist = []
    for year in YEARS:
        tmp = []
        for f in FILES:
            if plant in f.rsplit("_", 3)[0] and (str(year) in f.rsplit("_", 3)[1]):
                adf = read_df(f).drop("Datum bis", axis=1)
                adf['Datum von'] = pd.to_datetime(adf['Datum von'], format='%d.%m.%Y %H:%M')
                #print(f.rsplit("_", 3)[1])
                tmp.append(adf)
        dflist.append(tmp)
        tmp = []
    dfdict[plant] = dflist

In [10]:
#dfdict['Knapsack_Gas_I']

In [33]:
#dflist

In [12]:
rename_dict = {"Datum von": "produced_at", "Generation_DE GKM Block 6 [MW] Originalauflösungen": "SEE909002940690", "Generation_DE GKM Block 9 [MW] Originalauflösungen": "SEE988477355802",
              "Generation_DE Knapsack I [MW] Originalauflösungen": "SEE905192478234","Generation_DE Kraftwerk Knapsack II [MW] Originalauflösungen": "SEE903983424590"}

In [13]:
plant_mapper = {"Gro_kraftwerk_Mannheim": "06-08-2948214", "Knapsack_Gas_I": "06-05-300-9046030"}

In [34]:
for plant in plantlist:
    combined_df = pd.DataFrame()
    anotherlist = []
    for dfs in dfdict[plant]:
        if dfs == []:
            continue
        tmpdf = pd.concat(dfs, axis=1)
        anotherlist.append(tmpdf)
    nextlist = list(map(lambda x: x.iloc[:, np.setdiff1d(np.arange(len(x.columns)), 2)], anotherlist)) # reference: https://stackoverflow.com/a/68113529
    combined_df = pd.concat(nextlist, axis=0)
    final_df = combined_df.rename(columns=rename_dict)
    #final_df = final_df.replace('-', 0)
    final_df.fillna(0, inplace=True)
    #final_df[final_df.columns[2:]] = final_df[final_df.columns[2:]].astype(float)
    final_df[final_df.columns[1:]] = final_df[final_df.columns[1:]].astype(int)
    final_df.to_csv("../" + plant_mapper[plant] + ".csv", index=False)

In [31]:
plant_mapper[plant]

'06-05-300-9046030'

In [30]:
final_df

,produced_at,SEE903983424590,SEE905192478234
0,2015-01-01 00:00:00,0,0
1,2015-01-01 01:00:00,0,0
2,2015-01-01 02:00:00,0,0
3,2015-01-01 03:00:00,0,0
4,2015-01-01 04:00:00,0,0
...,...,...,...
8779,2024-12-31 19:00:00,0,171
8780,2024-12-31 20:00:00,0,0
8781,2024-12-31 21:00:00,0,0
8782,2024-12-31 22:00:00,0,0


In [16]:
'''
combined_df = pd.DataFrame()
anotherlist = []
for dfs in dflist:
    if dfs == []:
        continue
    tmpdf = pd.concat(dfs, axis=1)
    anotherlist.append(tmpdf)
nextlist = list(map(lambda x: x.iloc[:, np.setdiff1d(np.arange(len(x.columns)), 2)], anotherlist)) # reference: https://stackoverflow.com/a/68113529
combined_df = pd.concat(nextlist, axis=0)
final_df = combined_df.rename(columns=rename_dict)
final_df = final_df.replace("-", 0)
final_df = final_df.replace(",", '.')
final_df[final_df.columns[2:]] = final_df[final_df.columns[2:]].astype(int)
final_df.to_csv("../06-08-2948214.csv", index=False)
'''

'\ncombined_df = pd.DataFrame()\nanotherlist = []\nfor dfs in dflist:\n    if dfs == []:\n        continue\n    tmpdf = pd.concat(dfs, axis=1)\n    anotherlist.append(tmpdf)\nnextlist = list(map(lambda x: x.iloc[:, np.setdiff1d(np.arange(len(x.columns)), 2)], anotherlist)) # reference: https://stackoverflow.com/a/68113529\ncombined_df = pd.concat(nextlist, axis=0)\nfinal_df = combined_df.rename(columns=rename_dict)\nfinal_df = final_df.replace("-", 0)\nfinal_df = final_df.replace(",", \'.\')\nfinal_df[final_df.columns[2:]] = final_df[final_df.columns[2:]].astype(int)\nfinal_df.to_csv("../06-08-2948214.csv", index=False)\n'

In [17]:
final_df

,produced_at,SEE903983424590,SEE905192478234
0,2015-01-01 00:00:00,0,0
1,2015-01-01 01:00:00,0,0
2,2015-01-01 02:00:00,0,0
3,2015-01-01 03:00:00,0,0
4,2015-01-01 04:00:00,0,0
...,...,...,...
8779,2024-12-31 19:00:00,0,171
8780,2024-12-31 20:00:00,0,0
8781,2024-12-31 21:00:00,0,0
8782,2024-12-31 22:00:00,0,0


In [18]:
#anotherlist[0]

In [19]:
nextlist = list(map(lambda x: x.iloc[:, np.setdiff1d(np.arange(len(x.columns)), 2)], anotherlist)) # reference: https://stackoverflow.com/a/68113529

In [20]:
#nextlist[0]

In [21]:
combined_df = pd.concat(nextlist, axis=0)

In [22]:
#smardmapper = {"Gro_kraftwerk_Mannheim": ""}

In [23]:
list(combined_df)

['Datum von',
 'Generation_DE Kraftwerk Knapsack II [MW] Originalauflösungen',
 'Generation_DE Knapsack I [MW] Originalauflösungen']

In [24]:
final_df = combined_df.rename(columns=rename_dict)

In [25]:
final_df = final_df.replace('-', 0)

In [26]:
final_df.to_csv("../06-08-2948214.csv", index=False)

In [27]:
df1

NameError: name 'df1' is not defined

In [28]:
df1 = dflist[0]
df2 = dflist[1]

In [30]:
df1['Datum von'] = pd.to_datetime(df1['Datum von'], format="%d.%m.%Y %H:%M", utc=True)
df2['Datum von'] = pd.to_datetime(df2['Datum von'], format="%d.%m.%Y %H:%M", utc=True)

TypeError: list indices must be integers or slices, not str

In [31]:
df2

[               Datum von  \
 0    2016-01-01 00:00:00   
 1    2016-01-01 01:00:00   
 2    2016-01-01 02:00:00   
 3    2016-01-01 03:00:00   
 4    2016-01-01 04:00:00   
 ...                  ...   
 8779 2016-12-31 19:00:00   
 8780 2016-12-31 20:00:00   
 8781 2016-12-31 21:00:00   
 8782 2016-12-31 22:00:00   
 8783 2016-12-31 23:00:00   
 
       Generation_DE Kraftwerk Knapsack II [MW] Originalauflösungen  
 0                                                   0.0             
 1                                                   0.0             
 2                                                   0.0             
 3                                                   0.0             
 4                                                   0.0             
 ...                                                 ...             
 8779                                                0.0             
 8780                                                0.0             
 8781               

In [32]:
df3 = pd.concat([df1, df2], axis=1)

TypeError: cannot concatenate object of type '<class 'list'>'; only Series and DataFrame objs are valid

In [33]:
df3

NameError: name 'df3' is not defined

In [34]:
df4 = df3.drop(df3.columns[2], axis=1)

NameError: name 'df3' is not defined

In [ ]:
df4

In [ ]:
#dflist

In [ ]:
tmp = FILES[0]
tmp.rsplit("_", 3)

In [ ]:
FILES

In [ ]:
df1 = read_df(plantlist[0], 2023)

In [ ]:
df1 = pd.read_csv(plantname + "_" + year + "01010000_" + year+1 + "12312359_Stunde.csv", sep=';')
df2 = pd.read_csv("Gro_kraftwerk_Mannheim_202301010000_202412312359_Stunde(1).csv")
df3 = pd.read_csv("Gro_kraftwerk_Mannheim_202301010000_202412312359_Stunde(2).csv")

In [ ]:
df1